Using the Stage 1 to extract the Transmission Map and Atmospheric Light

In [11]:
import sys 

sys.path.append("..")

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import math
import torch.backends.cudnn as cudnn
import random
from mamba_ssm import Mamba

from thop import clever_format
from ptflops import get_model_complexity_info

import os
os.chdir("/workspace/dehazing")

In [12]:
def set_seed(seed):
    """Sets the seed for reproducibility across random, numpy, and torch."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

        cudnn.deterministic = True
        cudnn.benchmark = False

### Auxiliary Code

In [13]:
def get_pad_layer(pad_type):
    if(pad_type in ['refl','reflect']):
        PadLayer = nn.ReflectionPad2d
    elif(pad_type in ['repl','replicate']):
        PadLayer = nn.ReplicationPad2da
    elif(pad_type=='zero'):
        PadLayer = nn.ZeroPad2d
    else:
        print(f'Pad type [{pad_type}] not recognized')
    return PadLayer


class AntiAlias_Downsample(nn.Module):
    def __init__(self, channels, pad_type = 'reflect', filt_size = 3, 
                        stride = 2, pad_off = 0):
        super(AntiAlias_Downsample, self).__init__()
        self.filt_size = filt_size
        self.pad_off = pad_off
        self.pad_type = pad_type

        # Asymmetric padding (round up at the top and round down at the bottom)
        # Perfect when kernel size is 2 
        self.pad_sizes = [int(1. * (filt_size - 1) / 2), int(np.ceil(1. * (filt_size - 1) / 2)),
                          int(1. * (filt_size - 1) / 2), int(np.ceil(1. * (filt_size - 1) / 2))]
        self.pad_sizes = [pad_size + pad_off for pad_size in self.pad_sizes]
        self.stride = stride 
        self.off = int((self.stride - 1) / 2.)
        self.channels = channels 

        # Define the binomial filter weights
        if(self.filt_size==1):
            a = np.array([1.,])
        elif(self.filt_size==2):
            a = np.array([1., 1.])
        elif(self.filt_size==3):
            a = np.array([1., 2., 1.])
        elif(self.filt_size==4):    
            a = np.array([1., 3., 3., 1.])
        elif(self.filt_size==5):    
            a = np.array([1., 4., 6., 4., 1.])
        elif(self.filt_size==6):    
            a = np.array([1., 5., 10., 10., 5., 1.])
        elif(self.filt_size==7):    
            a = np.array([1., 6., 15., 20., 15., 6., 1.])
            
        # Create a 2D filter by taking the outer product of the 1D filter
        filt = torch.tensor(a[:, None] * a[None, :], dtype = torch.float32)
        filt = filt / torch.sum(filt) # Normalize

        # Reshape to (out_channels, in_channels/groups, kH, kW) for 
        # depthwise convolution
        filt = filt.view(1, 1, filt_size, filt_size)
        filt = filt.repeat(channels, 1, 1, 1)

        # Register as a buffer so PyTorch knows these are NOT trainable parameters
        self.register_buffer('filt', filt)
        self.pad = get_pad_layer(pad_type)(self.pad_sizes)

    def forward(self, inp):
        if (self.filt_size == 1):
            if (self.pad_off == 0):
                return inp[:, :, ::self.stride, ::self.stride] 
            else:
                return self.pad(inp)[:, :, ::self.stride, ::self.stride] 

        else:
            return F.conv2d(self.pad(inp), self.filt, stride = self.stride, groups = inp.shape[1])

In [14]:
class LocalFeatureExtractor(nn.Module):
    """
    Refined for Mamba Block Integration.
    Focuses on edge-preservation and local consistency.
    """
    def __init__(self, dim, kernel_size=3, dilation=1):
        super().__init__()
        padding = (dilation * (kernel_size - 1)) // 2
        
        # We use a Depthwise-Pointwise structure to keep it fast
        self.conv = nn.Sequential(
            nn.Conv2d(dim, dim, kernel_size=kernel_size, padding=padding, 
                      dilation=dilation, groups=dim), # Local spatial context
            nn.BatchNorm2d(dim),
            nn.SiLU(),
            nn.Conv2d(dim, dim, kernel_size=1), # Inter-channel communication
            nn.BatchNorm2d(dim)
        )

    def forward(self, x):
        # We add the input back (Residual) so that even if the gate is 
        # closed, the original features aren't lost.
        return x + self.conv(x)

In [15]:
class PhysBiMambaBlock(nn.Module):
    """
    Bidirectional Mamba Block (BiMamba)
    Scans the image Forward AND Backward so the top-left pixel
    can 'see' the bottom-right pixel.|
    """
    def __init__(self, dim, dropout = 0.05):
        super().__init__()
        self.norm = nn.LayerNorm(dim)

        # Note: In true VMamba, they share the input projection layer to save memory. 
        # But keeping 4 separate Mambas is fine if you have the GPU RAM.
        self.mamba_h_fwd = Mamba(d_model=dim, d_state=16, d_conv=4, expand=2)
        self.mamba_h_bwd = Mamba(d_model=dim, d_state=16, d_conv=4, expand=2)
        self.mamba_v_fwd = Mamba(d_model=dim, d_state=16, d_conv=4, expand=2)
        self.mamba_v_bwd = Mamba(d_model=dim, d_state=16, d_conv=4, expand=2)
        
        # Fuses Fwd+Bwd direction
        self.fusion_proj = nn.Linear(dim, dim)

        # Smoothing to explicitly destroy 1D streaking artifacts
        self.spatial_smoothing = nn.Conv2d(dim, dim, kernel_size=3, padding=1, groups=dim)
        
        self.local_conv = LocalFeatureExtractor(dim, kernel_size=3, dilation=1)
        
        # Optional: A Gate to let the network choose emphasis
        self.mixer = nn.Sequential(
            nn.Linear(dim * 2, dim),
            nn.Sigmoid()
        )
        
        # Initialize mixer bias negatively so local_conv is favored early in training
        nn.init.constant_(self.mixer[0].bias, -1.0)
        
        self.out_proj = nn.Linear(dim, dim)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, t_emb=None):
        B, C, H, W = x.shape
        residual = x
        
        x_flat = x.flatten(2).transpose(1, 2)
        x_norm = self.norm(x_flat)

        if t_emb is not None:
            scale, shift = t_emb.chunk(2, dim=1)
            x_norm = x_norm * (1 + scale.unsqueeze(1)) + shift.unsqueeze(1)

        # ---------------------------------------------------------
        # 2. HORIZONTAL SCANS (Raster Order)
        # ---------------------------------------------------------
        # Forward ->
        out_h_fwd = self.mamba_h_fwd(x_norm)
        
        # Backward <-
        x_flip = torch.flip(x_norm, dims=[1])
        out_h_bwd = self.mamba_h_bwd(x_flip)
        out_h_bwd = torch.flip(out_h_bwd, dims=[1]) # Flip back

        # ---------------------------------------------------------
        # 3. VERTICAL SCANS (Column-Major Order)
        # ---------------------------------------------------------
        # Reshape to Image -> Transpose (Swap H and W) -> Flatten
        # Result: (B, W*H, C). Now 'neighbors' in seq are vertical neighbors.
        x_v_img = x_norm.view(B, H, W, C).permute(0, 2, 1, 3) 
        x_v_flat = x_v_img.flatten(1, 2)
        
        # Down v
        out_v_fwd = self.mamba_v_fwd(x_v_flat)
        
        # Up ^
        x_v_flip = torch.flip(x_v_flat, dims=[1])
        out_v_bwd = self.mamba_v_bwd(x_v_flip)
        out_v_bwd = torch.flip(out_v_bwd, dims=[1])
        
        # Un-Transpose Vertical Outputs back to Horizontal Order
        # (B, W*H, C) -> (B, W, H, C) -> (B, H, W, C) -> (B, L, C)
        out_v_fwd = out_v_fwd.view(B, W, H, C).permute(0, 2, 1, 3).flatten(1, 2)
        out_v_bwd = out_v_bwd.view(B, W, H, C).permute(0, 2, 1, 3).flatten(1, 2)
        
        ## ---------------------------------------------------------
        # 4. Global Fusion
        # ---------------------------------------------------------
        # Combine all 4 views of the image
        # WE are treating the outputs of the 4 directions seperate.
        # It mixes the channels of the 4 scans for a single pixel, but it does not communicate with neighboring pixels. 
        # The model simply overlays the horizontal streaks and vertical streaks on top of each other.
        # global_feat = self.fusion_linear(
        #     torch.cat([out_h_fwd, out_h_bwd, out_v_fwd, out_v_bwd], dim=-1)
        # )

        global_feat = out_h_fwd + out_h_bwd + out_v_fwd + out_v_bwd
        global_feat = self.fusion_proj(global_feat)

        # ---------------------------------------------------------
        # 5. NEW: Spatial Smoothing to remove streaks
        # ---------------------------------------------------------
        global_feat_img = global_feat.transpose(1, 2).view(B, C, H, W)
        global_feat_img = self.spatial_smoothing(global_feat_img)
        global_feat = global_feat_img.flatten(2).transpose(1, 2)

        # ---------------------------------------------------------
        # 6. Local Branch (Conv)
        # ---------------------------------------------------------
        # Reshape for Conv2d
        x_img_norm = x_norm.transpose(1, 2).view(B, C, H, W)
        local_feat = self.local_conv(x_img_norm)
        local_feat = local_feat.flatten(2).transpose(1, 2)

        
        # ---------------------------------------------------------
        # 6. Gated Output
        # ---------------------------------------------------------
        combined = torch.cat([global_feat, local_feat], dim=-1)
        z = self.mixer(combined)
        
        fused = global_feat * z + local_feat * (1 - z)
        
        x_out = self.out_proj(fused)
        
        # Reshape to (B, C, H, W) for residual add
        x_out = x_out.transpose(1, 2).view(B, C, H, W)
        x_out = self.dropout(x_out)
        
        return residual + x_out


### Official Implementations

In [16]:
import torch
import torch.nn as nn
import torch.nn.functional as F


In [17]:
class BilinearUpsample(nn.Module):
    """
    Used by BOTH variants. Guarantees no checkerboard artifacts during decoding.
    """
    def __init__(self, dim_in, dim_out):
        super().__init__()
        self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        self.conv = nn.Conv2d(dim_in, dim_out, kernel_size=3, stride=1, padding=1)

    def forward(self, x):
        return self.conv(self.up(x))

In [18]:
class VariantA_StandardDownsample(nn.Module):
    """
    Standard stride=2 convolution. 
    Prone to aliasing and shift-variance (ignores Nyquist theorem).
    """
    def __init__(self, dim_in, dim_out):
        super().__init__()
        # Strided convolution drops 75% of pixels abruptly
        self.down = nn.Conv2d(dim_in, dim_out, kernel_size=4, stride=2, padding=1)

    def forward(self, x):
        return self.down(x)

In [19]:
class VariantB_AntiAliasedDownsample(nn.Module):
    """
    Anti-Aliased Downsampling (BlurPool) based on Richard Zhang's paper.
    Preserves shift-invariance and prevents high-frequency aliasing.
    """
    def __init__(self, dim_in, dim_out):
        super().__init__()
        # 1. Feature Mixing (Stride 1 preserves all spatial information)
        self.conv = nn.Conv2d(dim_in, dim_out, kernel_size=3, stride=1, padding=1)
        # 2. Anti-aliased spatial reduction (Low-pass filter + subsampling)
        # NOTE: Make sure your AntiAlias_Downsample class is defined in the script!
        self.aa_down = AntiAlias_Downsample(channels=dim_out, filt_size=3, stride=2)

    def forward(self, x):
        return self.aa_down(self.conv(x))

In [78]:
class BReLU(nn.Module):
    def __init__(self, t_min=0.0, t_max=1.0):
        super().__init__()
        self.t_min = t_min
        self.t_max = t_max

    def forward(self, x):
        return torch.clamp(x, self.t_min, self.t_max)

In [68]:
class AblationPhysicsEstimator(nn.Module):
    """
    Variants Summary:
    A: Standard CNN (Stride-2 Conv Downsampling)
    B: Physical Mamba + Anti-Aliased Downsampling
    C: Anti-Aliased CNN (No Mamba, isolate BlurPool effect)
    D: Dilated AA-CNN (Anti-Aliased Downsampling + Dilated Bottleneck)
    """
    def __init__(self, variant='D', in_channels=3, base_dim=32):
        super().__init__()
        self.variant = variant
        
        # Initial Encoder
        self.init_conv = nn.Conv2d(in_channels, base_dim, kernel_size=3, padding=1)
        self.enc1 = nn.Conv2d(base_dim, base_dim, kernel_size=3, padding=1)
        self.enc2 = nn.Conv2d(base_dim * 2, base_dim * 2, kernel_size=3, padding=1)

        # Logic for Downsampling and Bottleneck
        if variant == 'A':
            self.down1 = VariantA_StandardDownsample(base_dim, base_dim * 2)
            self.down2 = VariantA_StandardDownsample(base_dim * 2, base_dim * 4)

            self.bottleneck = nn.Conv2d(base_dim * 4, base_dim * 4, kernel_size=3, padding=1)
        elif variant == 'B':
            self.down1 = VariantB_AntiAliasedDownsample(base_dim, base_dim * 2)
            self.down2 = VariantB_AntiAliasedDownsample(base_dim * 2, base_dim * 4)

            self.bottleneck = PhysBiMambaBlock(dim=base_dim * 4)
            
        else:
            # Variants B, C, D all use Anti-Aliased Downsampling
            self.down1 = VariantB_AntiAliasedDownsample(base_dim, base_dim * 2)
            self.down2 = VariantB_AntiAliasedDownsample(base_dim * 2, base_dim * 4)
            
            
            if variant == 'C':
                self.bottleneck = nn.Conv2d(base_dim * 4, base_dim * 4, kernel_size=3, padding=1)
            elif variant == 'D':
                # Use Dilation=2 for larger receptive field without sequential artifacts
                self.bottleneck = LocalFeatureExtractor(base_dim * 4, dilation=2)
            else:
                raise ValueError("Variant must be A, B, C, or D")

        # Atmospheric Light (A) Head
        self.A_head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(base_dim * 4, 3), nn.Sigmoid() 
        )
        
        # Decoder with Skip Connections
        self.up1 = BilinearUpsample(base_dim * 4, base_dim * 2)
        self.dec1_conv = nn.Conv2d(base_dim * 4, base_dim * 2, kernel_size=1) 
        self.dec1 = nn.Conv2d(base_dim * 2, base_dim * 2, kernel_size=3, padding=1)
        
        self.up2 = BilinearUpsample(base_dim * 2, base_dim)
        self.dec2_conv = nn.Conv2d(base_dim * 2, base_dim, kernel_size=1)
        self.dec2 = nn.Conv2d(base_dim, base_dim, kernel_size=3, padding=1)
        
        # Transmission (t) Head
        self.t_head = nn.Sequential(nn.Conv2d(base_dim, 1, kernel_size=3, padding=1), nn.Sigmoid())

    def forward(self, x):
        # Initial Encoder + Activation
        x = F.relu(self.init_conv(x))
        e1 = F.relu(self.enc1(x))
        
        d1 = self.down1(e1)

        # Second Encoder Stage + Activation
        e2 = F.relu(self.enc2(d1))
        d2 = self.down2(e2)

        # Bottleneck
        b = F.relu(self.bottleneck(d2))
        
        A = self.A_head(b).view(-1, 3, 1, 1)

        # Decoder Stage 1
        u1 = torch.cat([self.up1(b), e2], dim=1)
        u1 = F.relu(self.dec1_conv(u1))
        u1 = F.relu(self.dec1(u1))
        
        # Decoder Stage 2
        u2 = torch.cat([self.up2(u1), e1], dim=1)
        u2 = F.relu(self.dec2_conv(u2))
        u2 = F.relu(self.dec2(u2))

        # t-head already has Sigmoid
        return self.t_head(u2), A


In [69]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_a = AblationPhysicsEstimator(variant='A').to(device)
model_b = AblationPhysicsEstimator(variant='B').to(device)
model_c = AblationPhysicsEstimator(variant='C').to(device)
model_d = AblationPhysicsEstimator(variant='D').to(device)


a_macs_raw, a_params_raw = get_model_complexity_info(
    model_a, (3, 256, 256),
    as_strings=True,
    print_per_layer_stat=False, 
    verbose=False
)

b_macs_raw, b_params_raw = get_model_complexity_info(
    model_b, (3, 256, 256),
    as_strings=True,
    print_per_layer_stat=False, 
    verbose=False
)

c_macs_raw, c_params_raw = get_model_complexity_info(
    model_c, (3, 256, 256),
    as_strings=True,
    print_per_layer_stat=False, 
    verbose=False
)

d_macs_raw, d_params_raw = get_model_complexity_info(
    model_d, (3, 256, 256),
    as_strings=True,
    print_per_layer_stat=False, 
    verbose=False
)
print
print("Model A has")
print(f"PARAMS {a_params_raw}, GFLOPS {a_macs_raw}") 
print("Model B has")
print(f"PARAMS {b_params_raw}, GFLOPS {b_macs_raw}") 
print("Model C has")
print(f"PARAMS {c_params_raw}, GFLOPS {c_macs_raw}") 
print("Model D has")
print(f"PARAMS {d_params_raw}, GFLOPS {d_macs_raw}") 


Model A has
PARAMS 508.13 k, GFLOPS 6.9 GMac
Model B has
PARAMS 840.55 k, GFLOPS 7.99 GMac
Model C has
PARAMS 436.45 k, GFLOPS 8.24 GMac
Model D has
PARAMS 307.17 k, GFLOPS 7.71 GMac


Testing the Stage 1

#### New updates

In [79]:
class BReLU(nn.Module):
    def __init__(self, t_min=0.0, t_max=1.0):
        super().__init__()
        self.t_min = t_min
        self.t_max = t_max

    def forward(self, x):
        return torch.clamp(x, self.t_min, self.t_max)

We will test on the RESIDE INDOORS first

In [70]:
from data.utils import get_haze_transforms, partition_dataset
from torch.utils.data import Subset
from losses import CharbonnierLoss
import torch
import numpy as np
import pandas as pd
from PIL import Image
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import v2

In [71]:
class RESIDE_Indoor(Dataset):
    def __init__(self, dataset_path, transform=None):
        self.root_dir = Path(dataset_path)
        self.metadata_csv = pd.read_csv(self.root_dir / "metadata.csv")

        self.transform = transform
        self.data = []
        
        for idx, row in self.metadata_csv.iterrows():
            clean_path = self.root_dir / row["clear_image_path"]
            hazy_paths_str = row["hazy_image_paths"]
            hazy_image_paths = [
                path.strip()
                for path in hazy_paths_str.strip("[]").replace("'", "").split(",")
            ]
            list_hazy_paths = [
                self.root_dir / hazy_path for hazy_path in hazy_image_paths
            ]
            
            for hazy_path in list_hazy_paths:
                # --- NEW LOGIC: Deduce the Transmission Map Path ---
                # Example: hazy_path.name is "1_1_0.90179.png"
                hazy_filename = hazy_path.name
                parts = hazy_filename.split('_')
                
                # Reconstruct trans filename: "1_1.png"
                if len(parts) >= 2:
                    trans_filename = f"{parts[0]}_{parts[1]}.png"
                else:
                    trans_filename = hazy_filename # Fallback just in case
                
                trans_path = self.root_dir / "trans" / trans_filename
                # ---------------------------------------------------

                data_item = {
                    "index": idx, 
                    "clean": clean_path, 
                    "hazy": hazy_path,
                    "trans": trans_path # Store the trans path
                }

                self.data.append(data_item)

    def __repr__(self):
        return "RESIDE Indoor"

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        data_item = self.data[idx]
        clean_path = data_item["clean"]
        hazy_path = data_item["hazy"]
        trans_path = data_item["trans"]

        try:
            clean_img = Image.open(clean_path).convert("RGB")
            hazy_img = Image.open(hazy_path).convert("RGB")
            # Load transmission map as Grayscale ("L")
            trans_img = Image.open(trans_path).convert("L") 
        except FileNotFoundError:
            print(f"Error: Missing image file at {clean_path}, {hazy_path}, or {trans_path}. Skipping")
            return self.__getitem__((idx + 1) % len(self))

        if self.transform:
            # IMPORTANT WARNING: 
            # If your 'get_haze_transforms' function only expects 2 inputs, 
            # you must update it to accept and return 3 inputs!
            clean_img, hazy_img, trans_img = self.transform(clean_img, hazy_img, trans_img)
        else:
            # Fallback tensorization
            clean_img = (
                torch.as_tensor(np.array(clean_img)).permute(2, 0, 1).float() / 255.0
            )
            hazy_img = (
                torch.as_tensor(np.array(hazy_img)).permute(2, 0, 1).float() / 255.0
            )
            # Add channel dimension to grayscale image (H, W) -> (1, H, W)
            trans_img = (
                torch.as_tensor(np.array(trans_img)).unsqueeze(0).float() / 255.0
            )

        return hazy_img, clean_img, trans_img

In [72]:
def get_reside_indoor_transforms(resize_size: int = 256):
    """
    Returns both train and val transforms strictly tuned for RESIDE-INDOOR.
    Safely handles the optional 3rd 'trans' (transmission) map.
    """
    
    # 1. Base Formatting: Convert to tensor [0, 1] -> Normalize to [-1, 1]
    to_tensor_norm = v2.Compose([
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
    ])

    # 2. Geometric Sync: Applied equally to clean, hazy, and trans to preserve alignment
    geometric_sync = v2.Compose([
        v2.RandomCrop(resize_size, pad_if_needed=True),
        v2.RandomHorizontalFlip(p=0.5),
        v2.RandomVerticalFlip(p=0.5),
    ])

    # 3. Color Jitter: Tuned specifically for RESIDE-INDOOR lighting
    color_jitter = v2.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15, hue=0.01)

    # --- TRAIN TRANSFORM ---
    def train_transform(clean, hazy, trans=None):
        # A. Apply spatial changes synchronously
        if trans is not None:
            clean, hazy, trans = geometric_sync(clean, hazy, trans)
        else:
            clean, hazy = geometric_sync(clean, hazy)

        # B. Apply color distortion ONLY to the hazy image
        hazy = color_jitter(hazy)

        # C. Tensor & Normalization
        clean, hazy = to_tensor_norm(clean), to_tensor_norm(hazy)
        
        if trans is not None:
            # Trans stays [0, 1] for physics math, NO normalization!
            trans = v2.functional.to_dtype(v2.functional.to_image(trans), torch.float32, scale=True)
            return clean, hazy, trans
            
        return clean, hazy

    # --- VAL TRANSFORM ---
    def val_transform(clean, hazy, trans=None):
        # Validation just normalizes. NO cropping or flipping.
        clean, hazy = to_tensor_norm(clean), to_tensor_norm(hazy)
        
        if trans is not None:
            trans = v2.functional.to_dtype(v2.functional.to_image(trans), torch.float32, scale=True)
            return clean, hazy, trans
            
        return clean, hazy

    return train_transform, val_transform

In [73]:
set_seed(42)

resolution = 256
verbose = True
num_subset_samples = 500

train_transform, val_transform = get_reside_indoor_transforms(resize_size = resolution)

data_path = "dataset/indoor-training-set/"
dataset = RESIDE_Indoor(dataset_path=data_path)

indices = torch.randperm(len(dataset))[:num_subset_samples].tolist()
subset_dataset = Subset(dataset, indices)

train_dataset, val_dataset = partition_dataset(
    subset_dataset,
    train_transform, 
    val_transform,
    train_ratio = 0.8
)

print(f"Total Subset Size: {len(subset_dataset)}")
print(f"Training Set Size: {len(train_dataset)}")
print(f"Validation Set Size: {len(val_dataset)}")

Total Subset Size: 500
Training Set Size: 400
Validation Set Size: 100


### Testing 

If we use the Self-Supervised Setup (when there is no transmission light, 
we need to use the TV Loss since there is not depth map from scratch and also the Atmospheric Prior). Since in the synthetic dataset, it already has though transmission map so we can drop those 

In [74]:
import torch.fft 
import os
import torch
import torch.nn.functional as F
import torchvision
import torch.fft
from torch.utils.tensorboard import SummaryWriter
from tqdm.notebook import tqdm
from torchmetrics.image import PeakSignalNoiseRatio, StructuralSimilarityIndexMeasure


def get_sobel_edges(img):
    """ Helper to extract edges using Sobel filters """
    kernel_x = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], device=img.device).float().unsqueeze(0).unsqueeze(0)
    kernel_y = torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], device=img.device).float().unsqueeze(0).unsqueeze(0)
    
    # Repeat for channels if necessary, but transmission is 1-ch
    edges_x = F.conv2d(img, kernel_x, padding=1)
    edges_y = F.conv2d(img, kernel_y, padding=1)
    return torch.sqrt(edges_x**2 + edges_y**2 + 1e-6)


def get_fft_spectrum(img):
    """ Helper to visualize the frequency spectrum """
    # Compute 2D FFT and shift low frequencies to center
    fft = torch.fft.fft2(img)
    fft_shift = torch.fft.fftshift(fft)

    # Magnitude in log scale for visualization
    magnitude = torch.log(torch.abs(fft_shift) + 1e-6)

    # Normalize to [0, 1] for TensorBoard
    mag_min, mag_max = magnitude.min(), magnitude.max()

    return (magnitude - mag_min) / (mag_max - mag_min + 1e-6)


In [75]:
class CharbonnierLoss(nn.Module):
    def __init__(self, eps=1e-3):
        super(CharbonnierLoss, self). __init__()
        self.eps = eps

    def forward(self, x, y):
        diff = x - y
        loss = torch.mean(torch.sqrt(diff * diff + self.eps * self.eps))
        return loss


class Stage1_RESIDELoss_Ablation(nn.Module):
    """
    Optimized Ablation Loss: Uses Charbonnier for everything to ensure 
    stability across all variants (especially Mamba).
    """
    def __init__(self, w_t=10.0, w_recon=1.0, w_edge = 0.3):
        super().__init__()
        self.charbonnier = CharbonnierLoss()
        self.w_t = w_t
        self.w_recon = w_recon
        self.w_edge = w_edge

    def forward(self, pred_t_map, pred_A, gt_t_map, clean_img_01, hazy_img_01):
        # 1. Clamp for safety
        t_map_safe = torch.clamp(pred_t_map, min=0.01, max=1.0)
        pred_A_safe = torch.clamp(pred_A, min=0.0, max=1.0)

        # Scale inputs from [-1, 1] to [0, 1] if necessary
        clean_img_01 = (clean_img_01 + 1.0) / 2.0
        hazy_img_01 = (hazy_img_01 + 1.0) / 2.0
        
        # 2. Transmission Supervision (Charbonnier)
        loss_t = self.charbonnier(t_map_safe, gt_t_map)
        
        # 3. Physical Reconstruction
        # I = Jt + A(1-t)
        I_recon = clean_img_01 * t_map_safe + pred_A_safe.view(-1, 3, 1, 1) * (1.0 - t_map_safe)
        loss_recon = self.charbonnier(I_recon, hazy_img_01)

        # 4. Edge-Aware Loss (High-Frequency Supervision)
        pred_edges = get_sobel_edges(t_map_safe)
        gt_edges = get_sobel_edges(gt_t_map)
        loss_edge = self.charbonnier(pred_edges, gt_edges)
        
        total_loss = (self.w_t * loss_t) + (self.w_recon * loss_recon) + (self.w_edge * loss_edge)
        
        return total_loss, {
            "Total_Loss": total_loss.item(),
            "Transmission_Charb": loss_t.item(),
            "Reconstruction_Charb": loss_recon.item(),
            "Edge_Charb": loss_edge.item(),
            "Mean_Predicted_A": pred_A_safe.mean().item()
        }


In [61]:
# class EdgeLoss(nn.Module):
#     """
#     Computes the L1 loss between the spatial gradients of predictions and targets.
#     This explicitly forces the network to output sharp edges and destroys 1D Mamba streaks.
#     """
#     def __init__(self):
#         super().__init__()
#         # Sobel filters for horizontal and vertical edge detection
#         k_x = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]]).view(1, 1, 3, 3)
#         k_y = torch.tensor([[-1., -2., -1.], [0., 0., 0.], [1., 2., 1.]]).view(1, 1, 3, 3)
#         self.register_buffer('k_x', k_x)
#         self.register_buffer('k_y', k_y)

#     def forward(self, pred, target):
#         # Calculate spatial gradients
#         pred_grad_x = F.conv2d(pred, self.k_x, padding=1)
#         pred_grad_y = F.conv2d(pred, self.k_y, padding=1)
#         target_grad_x = F.conv2d(target, self.k_x, padding=1)
#         target_grad_y = F.conv2d(target, self.k_y, padding=1)
        
#         # L1 loss on the gradients
#         loss = F.l1_loss(pred_grad_x, target_grad_x) + F.l1_loss(pred_grad_y, target_grad_y)
#         return loss

# # Make sure you have your CharbonnierLoss defined somewhere in your file!
# # class CharbonnierLoss(nn.Module):
# #     def __init__(self, eps=1e-3): ...

# class Stage1_RESIDELoss_V2(nn.Module):
#     """
#     Upgraded Supervised Physics Loss for Stage 1.
#     Replaces MSE with Charbonnier and adds Edge Loss to ensure perfect geometry.
#     """
#     def __init__(self):
#         super().__init__()
#         # We replace MSE with Charbonnier for the transmission map
#         self.charbonnier_t = CharbonnierLoss() 
#         self.charbonnier_recon = CharbonnierLoss()
        
#         # New edge constraint
#         self.edge_loss = EdgeLoss()
        
#         # Adjusted weights for the multi-objective setup
#         self.w_t_charb = 5.0     # Base pixel-wise accuracy (down from 10.0 since we split it)
#         self.w_t_edge = 2.0      # Sharp transitions and streak-prevention
#         self.w_recon = 1.0       # Physical constraint for A

#     def forward(self, pred_t_map, pred_A, gt_t_map, clean_img_01, hazy_img_01):
#         # 1. Clamp for safety
#         t_map_safe = torch.clamp(pred_t_map, min=0.01, max=1.0)
#         pred_A_safe = torch.clamp(pred_A, min=0.0, max=1.0)

#         # Scale inputs from [-1, 1] to [0, 1] if necessary
#         clean_img_01 = (clean_img_01 + 1.0) / 2.0
#         hazy_img_01 = (hazy_img_01 + 1.0) / 2.0
        
#         # 2. Direct Supervision on t(x) using Charbonnier (Replacing MSE)
#         loss_t_charb = self.charbonnier_t(t_map_safe, gt_t_map)
        
#         # 3. Edge/Gradient Loss to enforce sharp geometry 
#         loss_t_edge = self.edge_loss(t_map_safe, gt_t_map)
        
#         # 4. Physical Reconstruction (Forces the network to figure out A)
#         I_recon = clean_img_01 * t_map_safe + pred_A_safe * (1.0 - t_map_safe)
#         loss_recon = self.charbonnier_recon(I_recon, hazy_img_01)
        
#         # 5. Total Loss Calculation
#         total_loss = (self.w_t_charb * loss_t_charb) + \
#                      (self.w_t_edge * loss_t_edge) + \
#                      (self.w_recon * loss_recon)
        
#         # 6. Return dict for TensorBoard tracking
#         return total_loss, {
#             "Stage1_RESIDE/Total_Loss": total_loss.item(),
#             "Stage1_RESIDE/Transmission_Charbonnier": loss_t_charb.item(),
#             "Stage1_RESIDE/Transmission_Edge": loss_t_edge.item(),
#             "Stage1_RESIDE/Reconstruction": loss_recon.item(),
#             "Physics/Mean_Predicted_T": t_map_safe.mean().item(),
#             "Physics/Mean_GT_T": gt_t_map.mean().item(),
#             "Physics/Mean_Predicted_A": pred_A_safe.mean().item()
#         }

In [62]:
# class Stage1_PhysicsLoss(nn.Module):
#     """
#     Self-Supervised Physics Loss for Stage 1.
#     Uses the Clear Image (J) and predicted (t, A) to reconstruct the Hazy Image (I).
#     """
#     def __init__(self):
#         super().__init__()
#         self.charbonnier = CharbonnierLoss() # L1 with epsilon for stability
        
#         # Weights for Stage 1
#         self.w_recon = 1.0
#         self.w_tv = 0.05   # Keep small so it doesn't over-blur t(x) edges
#         self.w_atm = 0.01  # Gentle push to keep A reasonable

#     def get_gradients(self, img):
#         dy = img[:, :, 1:, :] - img[:, :, :-1, :]
#         dx = img[:, :, :, 1:] - img[:, :, :, :-1]
#         return dy, dx
    
#     def forward(self, pred_t_map, pred_A, clean_img_01, hazy_img_01):
#         # 1. Clamp predictions for physical safety
#         t_map_safe = torch.clamp(pred_t_map, min=0.01, max=1.0)
#         pred_A_safe = torch.clamp(pred_A, min=0.0, max=1.0)
        
#         # 2. Physics Reconstruction Loss (The core of Stage 1)
#         # Equation: I = J * t + A * (1 - t)
#         I_reconstructed = clean_img_01 * t_map_safe + pred_A_safe * (1.0 - t_map_safe)
#         loss_recon = self.charbonnier(I_reconstructed, hazy_img_01)

#         # 3. Total Variation Regularization (Smoothness for depth map)
#         dy, dx = self.get_gradients(t_map_safe)
#         loss_tv = torch.mean(torch.abs(dy)) + torch.mean(torch.abs(dx))

#         # 4. Atmospheric Regularizer (Push A away from pure black)
#         loss_atm = torch.mean(F.relu(0.05 - pred_A_safe)) 

#         # Aggregate
#         total_loss = (self.w_recon * loss_recon) + \
#                      (self.w_tv * loss_tv) + \
#                      (self.w_atm * loss_atm)

#         # Return dict for TensorBoard
#         return total_loss, {
#             "Stage1/Total_Loss": total_loss.item(),
#             "Stage1/Reconstruction": loss_recon.item(),
#             "Stage1/TV_Smoothness": loss_tv.item(),
#             "Stage1/Atm_Prior": loss_atm.item(),
#             "Physics/Mean_T": t_map_safe.mean().item(), # Track average haze density
#             "Physics/Mean_A": pred_A_safe.mean().item()
#         }

In [63]:
train_loader = DataLoader(train_dataset, batch_size = 16, shuffle = True)
val_loader = DataLoader(val_dataset, batch_size = 16, shuffle = True)

Diagnostic Functions

In [76]:
def train_ablation_variant(variant_code, train_loader, val_loader,
                           num_epochs=30, lr=1e-4, accum_iter=4, 
                           w_t = 10.0, w_recon = 1.0, w_edge = 0.3):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # 1. Setup Variant Naming
    names = {'A': 'Standard_CNN', 'B': 'Mamba_AA', 'C': 'CNN_AA', 'D': 'Dilated_AA'}
    v_name = names.get(variant_code, f"Variant_{variant_code}")
    print(f"🚀 Starting Ablation: {v_name} | Effective Batch Size: {train_loader.batch_size * accum_iter}")
    
    # 2. Initialize Loggers & Metrics
    log_dir = os.path.join("runs", "Stage1_Ablation_New_TestWeights", v_name)
    writer = SummaryWriter(log_dir=log_dir)

    psnr_metric = PeakSignalNoiseRatio(data_range=1.0).to(device)
    ssim_metric = StructuralSimilarityIndexMeasure(data_range=1.0).to(device)

    # 3. Model & Optimization
    model = AblationPhysicsEstimator(variant=variant_code).to(device)
    criterion = Stage1_RESIDELoss_Ablation(w_t = w_t, w_recon = w_recon, w_edge = w_edge).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

    global_step = 0

    for epoch in range(num_epochs):
        # --- TRAINING PHASE ---
        model.train()
        train_iterator = tqdm(train_loader, desc=f"Train {v_name} Ep {epoch+1}", leave=True)
        optimizer.zero_grad()
        
        for i, (hazy, clean, trans_gt) in enumerate(train_iterator):
            hazy, clean, trans_gt = hazy.to(device), clean.to(device), trans_gt.to(device)

            pred_t, pred_A = model(hazy)
            loss, loss_dict = criterion(pred_t, pred_A, trans_gt, clean, hazy)

            # Gradient Accumulation
            (loss / accum_iter).backward()

            if (i + 1) % accum_iter == 0 or (i + 1) == len(train_loader):
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                optimizer.zero_grad()

            if global_step % 20 == 0:
                for k, v in loss_dict.items():
                    writer.add_scalar(f"Train_Loss/{k}", v, global_step)

            global_step += 1
            train_iterator.set_postfix(loss=f"{loss.item():.4f}")

        # Update Scheduler
        scheduler.step()
        writer.add_scalar("Hyperparams/Learning_Rate", scheduler.get_last_lr()[0], epoch)
        
        # --- VALIDATION & QUANTITATIVE METRICS ---
        model.eval()
        val_mse_accum = 0.0
        edge_psnr_accum = 0.0
        recon_psnr_accum = 0.0

        psnr_metric.reset()
        ssim_metric.reset()

        val_iterator = tqdm(val_loader, desc="Validating", leave=False)
        with torch.no_grad():
            for v_hazy, v_clean, v_trans in val_iterator:
                v_hazy, v_clean, v_trans = v_hazy.to(device), v_clean.to(device), v_trans.to(device)
                
                p_t, p_A = model(v_hazy)
                p_t = torch.clamp(p_t, 0.0, 1.0)
                p_A = torch.clamp(p_A, 0.0, 1.0)

                I_recon = v_clean * p_t + p_A.view(-1, 3, 1, 1) * (1.0 - p_t)
                I_recon = torch.clamp(I_recon, 0.0, 1.0)
                recon_mse = F.mse_loss(I_recon, v_hazy)
                recon_psnr_accum += 10 * torch.log10(1.0 / (recon_mse + 1e-8)).item()

                # DIAGNOSTIC 1: EDGE-PSNR (Confirm Metric Trap)
                # We measure error ONLY on the sharp edges of the Ground Truth
                v_edges = get_sobel_edges(v_trans)
                edge_mask = (v_edges > 0.1).float() 
                edge_mse = F.mse_loss(p_t * edge_mask, v_trans * edge_mask)
                edge_psnr_accum += 10 * torch.log10(1.0 / (edge_mse + 1e-8)).item()
                
                psnr_metric.update(p_t, v_trans)
                ssim_metric.update(p_t, v_trans)
                val_mse_accum += F.mse_loss(p_t, v_trans).item()

            num_batches = len(val_loader)
            avg_psnr = psnr_metric.compute()
            avg_ssim = ssim_metric.compute()
            avg_mse = val_mse_accum /num_batches
            avg_edge_psnr = edge_psnr_accum / num_batches
            avg_recon_psnr = recon_psnr_accum / num_batches

            # Log Scalars
            writer.add_scalar("Metrics/Val_MSE", avg_mse, epoch)
            writer.add_scalar("Metrics/Val_PSNR", avg_psnr, epoch)
            writer.add_scalar("Metrics/Val_SSIM", avg_ssim, epoch)
            writer.add_scalar("Diagnostics/Edge_PSNR", avg_edge_psnr, epoch)
            writer.add_scalar("Diagnostics/Recon_PSNR", avg_recon_psnr, epoch)

            # --- QUALITATIVE DIAGNOSTIC GRID ---
            # Get a static batch for visual consistency across epochs
            sample_hazy, sample_clean, sample_trans = next(iter(val_loader))
            s_h = sample_hazy[:4].to(device)
            s_c = sample_clean[:4].to(device)
            s_t_gt = sample_trans[:4].to(device)

            s_t_pred, s_A_pred = model(s_h)
            s_t_pred = torch.clamp(s_t_pred, 0, 1)

            # DIAGNOSTIC 2: Sobel Error (Metric Trap Visualization)
            # This shows exactly where the edges are "blurring"
            abs_err = torch.abs(s_t_pred - s_t_gt)
            err_edges = get_sobel_edges(abs_err).repeat(1, 3, 1, 1)

            # DIAGNOSTIC 3: FFT Comparison (Spectral Bias Visualization)
            fft_gt_vis = get_fft_spectrum(s_t_gt).repeat(1, 3, 1, 1)
            fft_pred_vis = get_fft_spectrum(s_t_pred).repeat(1, 3, 1, 1)

            # Reconstruction row
            s_recon = s_c * s_t_pred + s_A_pred.view(-1, 3, 1, 1) * (1.0 - s_t_pred)
            
            # Prepare rows for the grid
            vis_gt_t = s_t_gt.repeat(1, 3, 1, 1)
            vis_pred_t = s_t_pred.repeat(1, 3, 1, 1)
            
            # Combine into 6-row grid: 
            # 1. Hazy | 2. GT Map | 3. Pred Map | 4. Edge Error | 5. FFT GT | 6. FFT Pred
            full_vis_stack = torch.cat([
                s_h,            # Original Input
                vis_gt_t,       # Target
                vis_pred_t,     # Model Output
                err_edges,      # WHERE IT FAILED (Edges)
                fft_gt_vis,     # Frequency Fingerprint Target
                fft_pred_vis    # Frequency Fingerprint Model
            ], dim=0)

            grid = torchvision.utils.make_grid(full_vis_stack, nrow=4, normalize=True)
            writer.add_image(f'Diagnostic_Grid/{v_name}', grid, epoch)

            # Also log a separate reconstruction check
            recon_grid = torchvision.utils.make_grid(torch.cat([s_h, s_recon], dim=0), nrow=4, normalize=True)
            writer.add_image(f'Reconstruction_Check/{v_name}', recon_grid, epoch)

        if epoch == num_epochs - 1:
            os.makedirs("checkpoints", exist_ok=True)
            torch.save(model.state_dict(), f"checkpoints/{v_name}_final.pth")

    writer.close()
    print(f"✅ Finished {v_name} | PSNR: {avg_psnr:.2f} | Edge PSNR: {avg_edge_psnr:.2f}")
    return model

Variant A

In [50]:
train_loader = DataLoader(train_dataset, batch_size = 16, shuffle = True)
val_loader = DataLoader(val_dataset, batch_size = 16, shuffle = False)

train_ablation_variant('A', train_loader, val_loader,
                    num_epochs=30, lr=1e-3, accum_iter = 2, 
                       w_t = 1.0, w_recon = 1.0, w_edge = 0.5)

🚀 Starting Ablation: Standard_CNN | Effective Batch Size: 32


Train Standard_CNN Ep 1:   0%|          | 0/25 [00:00<?, ?it/s]

Validating:   0%|          | 0/7 [00:00<?, ?it/s]

Train Standard_CNN Ep 2:   0%|          | 0/25 [00:00<?, ?it/s]

Validating:   0%|          | 0/7 [00:00<?, ?it/s]

...

Validating:   0%|          | 0/7 [00:00<?, ?it/s]

Train Standard_CNN Ep 30:   0%|          | 0/25 [00:00<?, ?it/s]

Validating:   0%|          | 0/7 [00:00<?, ?it/s]

✅ Finished Standard_CNN | PSNR: 18.49 | Edge PSNR: 34.56


AblationPhysicsEstimator(
  (init_conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (enc1): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (enc2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (down1): VariantA_StandardDownsample(
    (down): Conv2d(32, 64, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
  )
  (down2): VariantA_StandardDownsample(
    (down): Conv2d(64, 128, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
  )
  (bottleneck): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (A_head): Sequential(
    (0): AdaptiveAvgPool2d(output_size=1)
    (1): Flatten(start_dim=1, end_dim=-1)
    (2): Linear(in_features=128, out_features=3, bias=True)
    (3): Sigmoid()
  )
  (up1): BilinearUpsample(
    (up): Upsample(scale_factor=2.0, mode='bilinear')
    (conv): Conv2d(128, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  )
  (dec1_conv): Conv2d(128, 64, kernel_size=(1, 1), stride=

Variant B

In [77]:
train_loader = DataLoader(train_dataset, batch_size = 8, shuffle = True)
val_loader = DataLoader(val_dataset, batch_size = 8, shuffle = False)

train_ablation_variant('B', train_loader, val_loader,
                    num_epochs=30, lr=1e-3, accum_iter = 4, 
                       w_t = 1.0, w_recon = 1.0, w_edge = 0.5)

🚀 Starting Ablation: Mamba_AA | Effective Batch Size: 32


Train Mamba_AA Ep 1:   0%|          | 0/50 [00:00<?, ?it/s]

Validating:   0%|          | 0/13 [00:00<?, ?it/s]

Train Mamba_AA Ep 2:   0%|          | 0/50 [00:00<?, ?it/s]

Validating:   0%|          | 0/13 [00:00<?, ?it/s]

...

Validating:   0%|          | 0/13 [00:00<?, ?it/s]

Train Mamba_AA Ep 30:   0%|          | 0/50 [00:00<?, ?it/s]

Validating:   0%|          | 0/13 [00:00<?, ?it/s]

✅ Finished Mamba_AA | PSNR: 20.26 | Edge PSNR: 35.29


AblationPhysicsEstimator(
  (init_conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (enc1): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (enc2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (down1): VariantB_AntiAliasedDownsample(
    (conv): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (aa_down): AntiAlias_Downsample(
      (pad): ReflectionPad2d((1, 1, 1, 1))
    )
  )
  (down2): VariantB_AntiAliasedDownsample(
    (conv): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (aa_down): AntiAlias_Downsample(
      (pad): ReflectionPad2d((1, 1, 1, 1))
    )
  )
  (bottleneck): PhysBiMambaBlock(
    (norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
    (mamba_h_fwd): Mamba(
      (in_proj): Linear(in_features=128, out_features=512, bias=False)
      (conv1d): Conv1d(256, 256, kernel_size=(4,), stride=(1,), padding=(3,), groups=256)
      (act): SiLU()
      (x_pro

In [21]:
train_loader = DataLoader(train_dataset, batch_size = 8, shuffle = True)
val_loader = DataLoader(val_dataset, batch_size = 8, shuffle = True)

train_ablation_variant('C', train_loader, val_loader,
                    num_epochs=30, lr=1e-4, accum_iter = 4)

🚀 Starting Ablation: CNN_AA | Effective Batch Size: 32


Train CNN_AA Ep 1:   0%|          | 0/50 [00:00<?, ?it/s]

Validating:   0%|          | 0/13 [00:00<?, ?it/s]

Train CNN_AA Ep 2:   0%|          | 0/50 [00:00<?, ?it/s]

Validating:   0%|          | 0/13 [00:00<?, ?it/s]

...

Validating:   0%|          | 0/13 [00:00<?, ?it/s]

Train CNN_AA Ep 30:   0%|          | 0/50 [00:00<?, ?it/s]

Validating:   0%|          | 0/13 [00:00<?, ?it/s]

✅ Finished CNN_AA | PSNR: 17.52 | Edge PSNR: 33.67


AblationPhysicsEstimator(
  (init_conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (enc1): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (enc2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (down1): VariantB_AntiAliasedDownsample(
    (conv): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (aa_down): AntiAlias_Downsample(
      (pad): ReflectionPad2d((1, 1, 1, 1))
    )
  )
  (down2): VariantB_AntiAliasedDownsample(
    (conv): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (aa_down): AntiAlias_Downsample(
      (pad): ReflectionPad2d((1, 1, 1, 1))
    )
  )
  (bottleneck): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (A_head): Sequential(
    (0): AdaptiveAvgPool2d(output_size=1)
    (1): Flatten(start_dim=1, end_dim=-1)
    (2): Linear(in_features=128, out_features=3, bias=True)
    (3): Sigmoid()
  )
  (up1): BilinearUpsample(
    (up): Upsampl

In [22]:
train_loader = DataLoader(train_dataset, batch_size = 8, shuffle = True)
val_loader = DataLoader(val_dataset, batch_size = 8, shuffle = True)

train_ablation_variant('D', train_loader, val_loader,
                    num_epochs=30, lr=1e-4, accum_iter = 4)

🚀 Starting Ablation: Dilated_AA | Effective Batch Size: 32


Train Dilated_AA Ep 1:   0%|          | 0/50 [00:00<?, ?it/s]

Validating:   0%|          | 0/13 [00:00<?, ?it/s]

Train Dilated_AA Ep 2:   0%|          | 0/50 [00:00<?, ?it/s]

Validating:   0%|          | 0/13 [00:00<?, ?it/s]

...

Validating:   0%|          | 0/13 [00:00<?, ?it/s]

Train Dilated_AA Ep 30:   0%|          | 0/50 [00:00<?, ?it/s]

Validating:   0%|          | 0/13 [00:00<?, ?it/s]

✅ Finished Dilated_AA | PSNR: 18.70 | Edge PSNR: 34.18


AblationPhysicsEstimator(
  (init_conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (enc1): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (enc2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (down1): VariantB_AntiAliasedDownsample(
    (conv): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (aa_down): AntiAlias_Downsample(
      (pad): ReflectionPad2d((1, 1, 1, 1))
    )
  )
  (down2): VariantB_AntiAliasedDownsample(
    (conv): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (aa_down): AntiAlias_Downsample(
      (pad): ReflectionPad2d((1, 1, 1, 1))
    )
  )
  (bottleneck): LocalFeatureExtractor(
    (conv): Sequential(
      (0): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(2, 2), dilation=(2, 2), groups=128)
      (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU()
      (3): Conv2d(128, 128, kernel_siz

### DCP Benchmarks

In [23]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class BatchedDCP(nn.Module):
    def __init__(self, patch_size=15, omega=0.95, top_p=0.001):
        super().__init__()
        self.patch_size = patch_size
        self.omega = omega
        self.top_p = top_p

    def get_dark_channel(self, I):
        # 1. Min over RGB channels
        min_c, _ = torch.min(I, dim=1, keepdim=True)
        # 2. Min over spatial patch
        pad = self.patch_size // 2
        dc = -F.max_pool2d(-min_c, kernel_size=self.patch_size, stride=1, padding=pad)
        return dc

    def get_atmospheric_light(self, I, dc):
        B, C, H, W = I.shape
        A = torch.zeros(B, C, 1, 1, device=I.device)
        num_pixels = H * W
        num_top = max(int(num_pixels * self.top_p), 1)
        
        for i in range(B):
            dc_flat = dc[i, 0].view(-1)
            I_flat = I[i].view(C, -1)
            _, indices = torch.topk(dc_flat, num_top)
            A[i, :, 0, 0] = torch.mean(I_flat[:, indices], dim=1)
        return A

    def forward(self, I):
        dc = self.get_dark_channel(I)
        A = self.get_atmospheric_light(I, dc)
        
        # Normalize by A and get dark channel of normalized image
        I_norm = I / (A + 1e-6)
        t_map = 1.0 - self.omega * self.get_dark_channel(I_norm)
        
        return torch.clamp(t_map, 0.1, 1.0), torch.clamp(A, 0.0, 1.0)

In [54]:
def run_dcp_benchmark_fixed(val_loader, log_path="runs/Benchmarks/Classical_DCP"):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    writer = SummaryWriter(log_dir=log_path)
    dcp_model = BatchedDCP().to(device)
    
    # Metrics
    psnr_m = PeakSignalNoiseRatio(data_range=1.0).to(device)
    ssim_m = StructuralSimilarityIndexMeasure(data_range=1.0).to(device)
    
    edge_psnr_accum = 0.0
    val_iterator = tqdm(val_loader, desc="Benchmarking DCP")

    # Use fixed seed for consistent comparison with Model Benches
    g = torch.Generator()
    g.manual_seed(42)
    fixed_loader = DataLoader(val_loader.dataset, batch_size=4, shuffle=True, generator=g)
    
    # 1. Grab the Fixed Batch for Visuals
    s_h, s_c, s_t_gt = next(iter(fixed_loader))
    s_h, s_c, s_t_gt = s_h.to(device), s_c.to(device), s_t_gt.to(device)

    # with torch.no_grad():
    #     # --- DCP Inference ---
    #     t_dcp, A_dcp = dcp_model(s_h)
    #     t_dcp = torch.clamp(t_dcp, 0.0, 1.0)
        
    #     # --- Physical Reconstruction PSNR (The "Physics Check") ---
    #     # Formula: I = J*t + A*(1-t)
    #     A_reshaped = A_dcp.view(-1, 3, 1, 1)
    #     s_h_recon = s_c * t_dcp + A_reshaped * (1.0 - t_dcp)
    #     s_h_recon = torch.clamp(s_h_recon, 0.0, 1.0)
        
    #     recon_mse = F.mse_loss(s_h_recon, s_h)
    #     recon_psnr_val = 10 * torch.log10(1.0 / (recon_mse + 1e-8))

    #     # --- Helper for Local Normalization ---
    #     def norm_01(tensor):
    #         return (tensor - tensor.min()) / (tensor.max() - tensor.min() + 1e-8)

    #     # --- Build Diagnostic Rows (Normalized individually to fix darkness) ---
    #     vis_gt_t = s_t_gt.repeat(1, 3, 1, 1)
    #     vis_pred_t = t_dcp.repeat(1, 3, 1, 1)

    #     # Fix Edge Error: Normalize so the edges are white and the rest is black
    #     abs_err = torch.abs(t_dcp - s_t_gt)
    #     err_edges = get_sobel_edges(abs_err).repeat(1, 3, 1, 1)
    #     err_edges = norm_01(err_edges) 

    #     # Fix FFT: This is the row that causes the most "darkness" if not normalized locally
    #     fft_gt_vis = norm_01(get_fft_spectrum(s_t_gt)).repeat(1, 3, 1, 1)
    #     fft_pred_vis = norm_01(get_fft_spectrum(t_dcp)).repeat(1, 3, 1, 1)

    #     # --- Stack the 8-Row Grid ---
    #     vis_stack = torch.cat([
    #         torch.clamp(s_h, 0, 1),       # Row 1: Hazy Input
    #         torch.clamp(s_c, 0, 1),       # Row 2: Clean GT
    #         vis_gt_t,                     # Row 3: GT Map
    #         vis_pred_t,                   # Row 4: DCP Map
    #         err_edges,                    # Row 5: Locally Normalized Edge Error
    #         fft_gt_vis,                   # Row 6: Locally Normalized FFT GT
    #         fft_pred_vis,                 # Row 7: Locally Normalized FFT DCP
    #         s_h_recon                     # Row 8: Reconstructed Hazy
    #     ], dim=0)

    #     # normalize=False is the CRITICAL fix here
    #     grid = torchvision.utils.make_grid(vis_stack, nrow=4, normalize=False)
    with torch.no_grad():
        s_t_pred, s_A_pred = dcp_model(s_h)
        s_t_pred = torch.clamp(s_t_pred, 0.0, 1.0)
        
        # --- PHYSICAL RECONSTRUCTION ---
        # I_recon = J*t + A*(1-t)
        A_reshaped = s_A_pred.view(-1, 3, 1, 1)
        s_h_recon = s_c * s_t_pred + A_reshaped * (1.0 - s_t_pred)
        s_h_recon = torch.clamp(s_h_recon, 0.0, 1.0)

        # 5. CALCULATE RECONSTRUCTION PSNR
        # This measures how well the model "re-synthesized" the hazy input
        recon_mse = F.mse_loss(s_h_recon, s_h)
        recon_psnr = 10 * torch.log10(1.0 / (recon_mse + 1e-8))

    # 6. Build Diagnostic Rows (Normalized individually to fix "Darkness" issue)
    vis_gt_t = s_t_gt.repeat(1, 3, 1, 1)
    vis_pred_t = s_t_pred.repeat(1, 3, 1, 1)

    # Edge Error Local Normalization
    abs_err = torch.abs(s_t_pred - s_t_gt)
    err_edges = get_sobel_edges(abs_err).repeat(1, 3, 1, 1)
    err_edges = err_edges / (err_edges.max() + 1e-8)

    # FFT Spectrum Local Normalization
    fft_gt_vis = get_fft_spectrum(s_t_gt).repeat(1, 3, 1, 1)
    fft_pred_vis = get_fft_spectrum(s_t_pred).repeat(1, 3, 1, 1)
    fft_gt_vis = fft_gt_vis / (fft_gt_vis.max() + 1e-8)
    fft_pred_vis = fft_pred_vis / (fft_pred_vis.max() + 1e-8)

    # 7. Stack the 8-Row Forensic Grid
    full_vis_stack = torch.cat([
        s_h,           # Row 1: Original Hazy (I)
        s_c,           # Row 2: Clean GT (J) 
        vis_gt_t,      # Row 3: Target Transmission (t_gt)
        vis_pred_t,    # Row 4: Predicted Transmission (t_pred)
        err_edges,     # Row 5: Normalized Edge Error
        fft_gt_vis,    # Row 6: Normalized FFT Target
        fft_pred_vis,  # Row 7: Normalized FFT Model
        s_h_recon      # Row 8: Reconstructed Hazy (The Physics Check)
    ], dim=0)

    # Final Grid (normalize=False prevents global brightness crushing)
    grid = torchvision.utils.make_grid(full_vis_stack, nrow=4, normalize=False)

    writer.add_image('benchmark_check/DCP_Baseline', grid, 0)
    writer.add_scalar("Benchmark/Reconstruction_PSNR", recon_psnr, 0)

    # --- Quantitative Pass over full val_loader ---
    for hazy, clean, trans_gt in val_iterator:
        hazy, trans_gt = hazy.to(device), trans_gt.to(device)
        t_out, _ = dcp_model(hazy)
        t_out = torch.clamp(t_out, 0.0, 1.0)
        
        psnr_m.update(t_out, trans_gt)
        ssim_m.update(t_out, trans_gt)
        
        # Edge PSNR Calculation
        v_edges = get_sobel_edges(trans_gt)
        edge_mask = (v_edges > 0.1).float()
        e_mse = F.mse_loss(t_out * edge_mask, trans_gt * edge_mask)
        edge_psnr_accum += 10 * torch.log10(1.0 / (e_mse + 1e-8)).item()

    # Final result logging
    final_psnr = psnr_m.compute()
    final_ssim = ssim_m.compute()
    final_edge_psnr = edge_psnr_accum / len(val_loader)

    writer.add_scalar("Benchmark/PSNR", final_psnr, 0)
    writer.add_scalar("Benchmark/SSIM", final_ssim, 0)
    writer.add_scalar("Benchmark/Edge_PSNR", final_edge_psnr, 0)
    
    print(f"📊 DCP BENCHMARK: PSNR: {final_psnr:.2f} | Recon PSNR: {recon_psnr:.2f} | Edge PSNR: {final_edge_psnr:.2f}")
    writer.close()

In [55]:
run_dcp_benchmark_fixed(val_loader)

Benchmarking DCP:   0%|          | 0/13 [00:00<?, ?it/s]

📊 DCP BENCHMARK: PSNR: 15.71 | Recon PSNR: 26.02 | Edge PSNR: 32.57


## Comparison

In [34]:
import torch
import torch.nn.functional as F
import torchvision
from torch.utils.tensorboard import SummaryWriter
from torch.utils.data import DataLoader
import os

def load_and_benchmark(variant_code, checkpoint_path, val_loader, log_base="runs/Benchmarks/Model_Checks"):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # 1. Setup Naming
    names = {'A': 'Standard_CNN', 'B': 'Mamba_AA', 'C': 'CNN_AA', 'D': 'Dilated_AA'}
    v_name = names.get(variant_code, f"Variant_{variant_code}")
    writer = SummaryWriter(log_dir=os.path.join(log_base, v_name))

    # 2. Initialize Model
    model = AblationPhysicsEstimator(variant=variant_code).to(device)
    state_dict = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(state_dict)
    model.eval()

    # 3. Get the EXACT SAME fixed batch (using a seed)
    g = torch.Generator()
    g.manual_seed(42) 
    fixed_loader = DataLoader(val_loader.dataset, batch_size=4, shuffle=True, generator=g)
    s_h, s_c, s_t_gt = next(iter(fixed_loader))
    
    s_h, s_c, s_t_gt = s_h.to(device), s_c.to(device), s_t_gt.to(device)

    # 4. Inference
    with torch.no_grad():
        s_t_pred, s_A_pred = model(s_h)
        s_t_pred = torch.clamp(s_t_pred, 0.0, 1.0)
        
        # --- PHYSICAL RECONSTRUCTION ---
        # I_recon = J*t + A*(1-t)
        A_reshaped = s_A_pred.view(-1, 3, 1, 1)
        s_h_recon = s_c * s_t_pred + A_reshaped * (1.0 - s_t_pred)
        s_h_recon = torch.clamp(s_h_recon, 0.0, 1.0)

        # 5. CALCULATE RECONSTRUCTION PSNR
        # This measures how well the model "re-synthesized" the hazy input
        recon_mse = F.mse_loss(s_h_recon, s_h)
        recon_psnr = 10 * torch.log10(1.0 / (recon_mse + 1e-8))

    # 6. Build Diagnostic Rows (Normalized individually to fix "Darkness" issue)
    vis_gt_t = s_t_gt.repeat(1, 3, 1, 1)
    vis_pred_t = s_t_pred.repeat(1, 3, 1, 1)

    # Edge Error Local Normalization
    abs_err = torch.abs(s_t_pred - s_t_gt)
    err_edges = get_sobel_edges(abs_err).repeat(1, 3, 1, 1)
    err_edges = err_edges / (err_edges.max() + 1e-8)

    # FFT Spectrum Local Normalization
    fft_gt_vis = get_fft_spectrum(s_t_gt).repeat(1, 3, 1, 1)
    fft_pred_vis = get_fft_spectrum(s_t_pred).repeat(1, 3, 1, 1)
    fft_gt_vis = fft_gt_vis / (fft_gt_vis.max() + 1e-8)
    fft_pred_vis = fft_pred_vis / (fft_pred_vis.max() + 1e-8)

    # 7. Stack the 8-Row Forensic Grid
    full_vis_stack = torch.cat([
        s_h,           # Row 1: Original Hazy (I)
        s_c,           # Row 2: Clean GT (J) 
        vis_gt_t,      # Row 3: Target Transmission (t_gt)
        vis_pred_t,    # Row 4: Predicted Transmission (t_pred)
        err_edges,     # Row 5: Normalized Edge Error
        fft_gt_vis,    # Row 6: Normalized FFT Target
        fft_pred_vis,  # Row 7: Normalized FFT Model
        s_h_recon      # Row 8: Reconstructed Hazy (The Physics Check)
    ], dim=0)

    # Final Grid (normalize=False prevents global brightness crushing)
    grid = torchvision.utils.make_grid(full_vis_stack, nrow=4, normalize=False)
    
    # 8. Log Everything
    writer.add_image(f'benchmark_check/{v_name}', grid, 0)
    writer.add_scalar("Benchmark/Reconstruction_PSNR", recon_psnr, 0)
    writer.add_scalar("Benchmark/Mean_A_Value", s_A_pred.mean(), 0)
    
    print(f"--- {v_name} BENCHMARK COMPLETE ---")
    print(f"PSNR (Hazy vs Recon): {recon_psnr.item():.2f} dB")
    print(f"Mean A-light: {s_A_pred.mean().item():.4f}")
    
    writer.close()

In [51]:
# Model A
checkpoint_A = "checkpoints/Standard_CNN_final.pth"
load_and_benchmark("A", checkpoint_A, val_loader, log_base="runs/Benchmarks/Model_Checks")

--- Standard_CNN BENCHMARK COMPLETE ---
PSNR (Hazy vs Recon): 25.56 dB
Mean A-light: 0.9269


In [81]:
# Model B
checkpoint_B = "checkpoints/Mamba_AA_final.pth"
load_and_benchmark("B", checkpoint_B, val_loader, log_base="runs/Benchmarks/Model_Checks")

--- Mamba_AA BENCHMARK COMPLETE ---
PSNR (Hazy vs Recon): 26.99 dB
Mean A-light: 0.9219


In [42]:
# Model C
checkpoint_C = "checkpoints/CNN_AA_final.pth"
load_and_benchmark("C", checkpoint_C, val_loader, log_base="runs/Benchmarks/Model_Checks")

--- CNN_AA BENCHMARK COMPLETE ---
PSNR (Hazy vs Recon): 24.35 dB
Mean A-light: 0.9241


In [43]:
# Model D
checkpoint_D = "checkpoints/Dilated_AA_final.pth"
load_and_benchmark("D", checkpoint_D, val_loader, log_base="runs/Benchmarks/Model_Checks")

--- Dilated_AA BENCHMARK COMPLETE ---
PSNR (Hazy vs Recon): 19.44 dB
Mean A-light: 0.7033


### Using the DehazeNet - Paradigm

In [83]:
import torch
import torch.nn as nn

class BReLU(nn.Module):
    """
    Bilateral Rectified Linear Unit (BReLU)
    Bounds the output strictly between t_min and t_max (typically 0 and 1).
    """
    def __init__(self, t_min=0.0, t_max=1.0):
        super(BReLU, self).__init__()
        self.t_min = t_min
        self.t_max = t_max

    def forward(self, x):
        return torch.clamp(x, min=self.t_min, max=self.t_max)


class Maxout(nn.Module):
    """
    Maxout Activation Unit.
    Splits the channels into groups of size `num_pieces` and takes the maximum 
    across the channel dimension to reduce dimensionality.
    """
    def __init__(self, num_pieces):
        super(Maxout, self).__init__()
        self.num_pieces = num_pieces

    def forward(self, x):
        # x shape: (Batch, Channels, Height, Width)
        b, c, h, w = x.shape
        # Group channels and take the maximum
        x = x.view(b, c // self.num_pieces, self.num_pieces, h, w)
        return torch.max(x, dim=2)[0]


In [ ]:
class EndToEndDehazeNet(nn.Module):
    def __init__(self, patch_size=15, top_p=0.001):
        super(EndToEndDehazeNet, self).__init__()
        
        # --- DCP Parameters for Atmospheric Light ---
        self.patch_size = patch_size
        self.top_p = top_p
        
        # --- DehazeNet CNN Layers for Transmission Map ---
        # 1. Feature Extraction
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=5, padding=2)
        self.maxout = Maxout(num_pieces=4) 
        
        # 2. Multi-scale Mapping
        self.conv2_3 = nn.Conv2d(in_channels=4, out_channels=16, kernel_size=3, padding=1)
        self.conv2_5 = nn.Conv2d(in_channels=4, out_channels=16, kernel_size=5, padding=2)
        self.conv2_7 = nn.Conv2d(in_channels=4, out_channels=16, kernel_size=7, padding=3)
        
        # 3. Local Extremum
        self.maxpool = nn.MaxPool2d(kernel_size=7, stride=1, padding=3)
        
        # 4. Non-linear Regression
        self.pad = nn.ZeroPad2d((2, 3, 2, 3)) 
        self.conv3 = nn.Conv2d(in_channels=48, out_channels=1, kernel_size=6, padding=0)
        self.brelu = BReLU(t_min=0.0, t_max=1.0)

    def get_dark_channel(self, I):
        """Calculates the dark channel prior of the image."""
        # 1. Min over RGB channels
        min_c, _ = torch.min(I, dim=1, keepdim=True)
        # 2. Min over spatial patch
        pad = self.patch_size // 2
        dc = -F.max_pool2d(-min_c, kernel_size=self.patch_size, stride=1, padding=pad)
        return dc

    def get_atmospheric_light(self, I, dc):
        """Estimates global atmospheric light based on top brightest pixels in the dark channel."""
        B, C, H, W = I.shape
        A = torch.zeros(B, C, 1, 1, device=I.device)
        num_pixels = H * W
        num_top = max(int(num_pixels * self.top_p), 1)
        
        for i in range(B):
            dc_flat = dc[i, 0].view(-1)
            I_flat = I[i].view(C, -1)
            _, indices = torch.topk(dc_flat, num_top)
            A[i, :, 0, 0] = torch.mean(I_flat[:, indices], dim=1)
        return A

    def forward(self, I):
        """
        I: Hazy input image tensor of shape (B, 3, H, W) in range [0, 1]
        Returns: Restored image J, Transmission Map t, Atmospheric Light A
        """
        
        # ==========================================
        # 1. Estimate Atmospheric Light (A)
        # ==========================================
        dc = self.get_dark_channel(I)
        A = self.get_atmospheric_light(I, dc)
        A = torch.clamp(A, 0.0, 1.0) # Sanity clamp
        
        # ==========================================
        # 2. Estimate Transmission Map (t) via CNN
        # ==========================================
        x = self.conv1(I)
        x = self.maxout(x)
        
        x3 = self.conv2_3(x)
        x5 = self.conv2_5(x)
        x7 = self.conv2_7(x)
        x = torch.cat([x3, x5, x7], dim=1) 
        
        x = self.maxpool(x)
        
        x = self.pad(x)
        x = self.conv3(x)
        t = self.brelu(x)
        
        # ==========================================
        # 3. Reconstruct Haze-Free Image (J)
        # ==========================================
        # To avoid division by zero or blowing up noise in very dense haze, 
        # we strictly clamp the transmission map to a minimum bound (typically 0.1)
        t_safe = torch.clamp(t, min=0.1)
        
        # Apply the atmospheric scattering model inversion
        J = (I - A) / t_safe + A
        
        # Clamp the final image back to valid RGB bounds [0, 1]
        J = torch.clamp(J, 0.0, 1.0)
        
        # Returning all three variables is highly recommended for calculating losses 
        # (e.g., if you want to apply a loss directly to J, and an edge loss to t)
        return J, t, A